In [2]:
import pandas as pd
import os
from glob import glob

# 原始数据路径
data_dir = '../data/raw/PAMAP2_Dataset/Protocol/'
all_files = glob(os.path.join(data_dir, '*.dat'))

# 按照 readme.pdf 定义 54 列名
columns = [
    'timestamp', 'activityID', 'heart_rate'
] + [f'hand_{i}' for i in range(1, 18)] \
  + [f'chest_{i}' for i in range(1, 18)] \
  + [f'ankle_{i}' for i in range(1, 18)]

df_list = []
for filepath in all_files:
    subject_id = os.path.basename(filepath).split('.')[0].replace('subject', '')
    df_temp = pd.read_csv(filepath, sep=' ', header=None, names=columns)
    df_temp['subject_id'] = subject_id
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)
print("合并完成，数据形状：", df_all.shape)

合并完成，数据形状： (2872533, 55)


In [3]:
# 手腕加速度（±16g）和陀螺仪列
wrist_acc_cols = ['hand_2', 'hand_3', 'hand_4']
wrist_gyro_cols = ['hand_8', 'hand_9', 'hand_10']
selected_cols = ['activityID', 'subject_id'] + wrist_acc_cols + wrist_gyro_cols

df_wrist = df_all[selected_cols].copy()

# 筛选健身动作 ID（跑步、跳绳、踢足球、骑车、走路）
fitness_ids = [5, 24, 20, 6, 4]
df_fitness = df_wrist[df_wrist['activityID'].isin(fitness_ids)].copy()

# 重命名列，以匹配项目要求
df_fitness.rename(columns={
    'hand_2': 'acc_x', 'hand_3': 'acc_y', 'hand_4': 'acc_z',
    'hand_8': 'gyro_x', 'hand_9': 'gyro_y', 'hand_10': 'gyro_z',
    'activityID': 'label'
}, inplace=True)

print("✅ 筛选完成，数据形状：", df_fitness.shape)

✅ 筛选完成，数据形状： (550920, 8)


In [4]:
import os

os.makedirs('../data/processed/', exist_ok=True)
df_fitness.to_csv('../data/processed/pamap2_processed.csv', index=False)
print("✅ 数据已保存到 data/processed/pamap2_processed.csv")

✅ 数据已保存到 data/processed/pamap2_processed.csv
